## 1. 라이브러리 로드

In [58]:
# 라이브러리 호출
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
import platform
import ast
from collections import Counter, defaultdict
import json

warnings.filterwarnings('ignore')

# 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # Mac
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

# 컬럼 너비 제한 해제
pd.set_option('display.max_colwidth', None)

---
## 2. 데이터 로드

In [59]:
df_platinum_Match = pd.read_csv('./raw_data/TFT_Platinum_MatchData.csv')
df_diamond_Match = pd.read_csv('./raw_data/TFT_Diamond_MatchData.csv')
df_master_Match = pd.read_csv('./raw_data/TFT_Master_MatchData.csv')
df_grand_master_Match = pd.read_csv('./raw_data/TFT_GrandMaster_MatchData.csv')
df_challenger_Match = pd.read_csv('./raw_data/TFT_Challenger_MatchData.csv')

df_champion_info = pd.read_csv('./raw_data/TFT_Champion_CurrentVersion.csv')
df_items_info = pd.read_csv('./raw_data/TFT_Item_CurrentVersion.csv')

In [60]:
# 매치관련 데이터가 담긴 딕셔너리 정의
match_data = {
    'platinum': df_platinum_Match,
    'diamond': df_diamond_Match,
    'master': df_master_Match,
    'grand_master': df_grand_master_Match,
    'challenger': df_challenger_Match,
}

# 각 티어별 테이블에 'Tier'정보가 담긴 컬럼 추가
for name, df in match_data.items():
    df['tier'] = name


# 모든 티어의 경기데이터가 담긴 데이터프레임 제작
# ignore_index=True: 데이터를 병합한 뒤, 새로운 인덱스 부여
df_all_match = pd.concat(match_data.values(), ignore_index=True)

df_all_match.head(3)

,gameId,gameDuration,level,lastRound,Ranked,ingameDuration,combination,champion,tier
0,KR_4291707834,1963.905273,6,27,5,1390.165771,"{'Cybernetic': 1, 'Demolitionist': 1, 'Infiltrator': 1, 'Rebel': 1, 'Set3_Brawler': 1, 'Set3_Celestial': 1, 'Set3_Void': 1, 'Sniper': 1}","{'Ziggs': {'items': [7], 'star': 1}, 'Ashe': {'items': [9], 'star': 1}, 'ChoGath': {'items': [6], 'star': 1}, 'Ekko': {'items': [1], 'star': 1}}",platinum
1,KR_4291707834,1963.905273,8,37,3,1891.282715,"{'Blaster': 1, 'Chrono': 1, 'Cybernetic': 4, 'Demolitionist': 1, 'Rebel': 1, 'Set3_Blademaster': 2, 'Set3_Brawler': 1, 'Set3_Sorcerer': 1, 'Set3_Void': 1, 'Valkyrie': 1, 'Vanguard': 2}","{'Ziggs': {'items': [24], 'star': 3}, 'Fiora': {'items': [37], 'star': 2}, 'Leona': {'items': [36, 24], 'star': 2}, 'Lucian': {'items': [], 'star': 2}, 'Vi': {'items': [5], 'star': 2}, 'Kayle': {'items': [], 'star': 2}, 'WuKong': {'items': [3, 67], 'star': 2}, 'VelKoz': {'items': [4], 'star': 2}}",platinum
2,KR_4291707834,1963.905273,6,25,7,1279.461060,"{'Blaster': 1, 'Cybernetic': 1, 'DarkStar': 2, 'Infiltrator': 1, 'Mercenary': 1, 'Set3_Blademaster': 1, 'Set3_Mystic': 1, 'Valkyrie': 1}","{'Fiora': {'items': [1], 'star': 1}, 'Shaco': {'items': [6], 'star': 1}, 'Karma': {'items': [4], 'star': 1}, 'MissFortune': {'items': [3], 'star': 1}}",platinum


---
## 3. 데이터 전처리
### 3-1. 중복행 제거

In [61]:
# 중복행 제거
duplicates = df_all_match[df_all_match.duplicated(keep=False)]

print(f"중복 제거 전: {len(df_all_match)}")
print(f"중복된 행 개수: {len(duplicates)}")

# 결과 확인
df_all_match = df_all_match.drop_duplicates()
print(f"\n중복 제거 후: {len(df_all_match)}")

중복 제거 전: 399998
중복된 행 개수: 80

중복 제거 후: 399930


### 3-2. 컬럼명 소문자로 통일

In [62]:
# 전체 컬럼명 소문자 통일
df_all_match.columns = df_all_match.columns.str.lower()

print(df_all_match.columns)

Index(['gameid', 'gameduration', 'level', 'lastround', 'ranked',
       'ingameduration', 'combination', 'champion', 'tier'],
      dtype='str')


### 3-3. ranked=0인 데이터 삭제

In [63]:
# ranked = 0이 포함된 경기 데이터 삭제
df_all_match_2 = df_all_match.copy()

# ranked==0인 행의 gameId 추출
drop_game_ids = df_all_match_2[df_all_match_2['ranked']==0]['gameid'].unique()

# 해당 gameId를 가진 행의 인덱스 추출 후 삭제
drop_idx = df_all_match_2[df_all_match_2['gameid'].isin(drop_game_ids)].index
df_all_match_2 = df_all_match_2.drop(index=drop_idx)

print(f"ranked = 0인 데이터가 포함된 경기 데이터 삭제하기 전: {df_all_match.shape}")
print(f"ranked = 0인 데이터가 포함된 경기 데이터 삭제한 후: {df_all_match_2.shape}")

ranked = 0인 데이터가 포함된 경기 데이터 삭제하기 전: (399930, 9)
ranked = 0인 데이터가 포함된 경기 데이터 삭제한 후: (399886, 9)


### 3-4. 경기시간 = 0인 데이터 삭제

In [64]:
# 전체 게임시간 = 0인 행의 GameId 추출
zero_duration_ids = set(df_all_match_2[df_all_match_2['gameduration'] == 0]['gameid'])

# 해당 gameId를 가진 행의 인덱스 추출 후 삭제
drop_idx_2 = df_all_match_2[df_all_match_2['gameid'].isin(zero_duration_ids)].index
df_all_match_2 = df_all_match_2.drop(index=drop_idx_2)

print(f"gameduration = 0인 데이터가 포함된 경기 데이터 삭제한 후: {df_all_match_2.shape}")

gameduration = 0인 데이터가 포함된 경기 데이터 삭제한 후: (399886, 9)


### 3-5. 경기 참여 유저 수가 8명 미만인 게임 삭제

In [65]:
# 게임 ID별 플레이어 수 정보 추가
df_all_match_2['player_cnt'] = df_all_match_2['gameid'].map(df_all_match_2['gameid'].value_counts())

# player_cnt < 8인 행의 gameId 추출
drop_game_ids_3 = df_all_match_2[df_all_match_2['player_cnt'] < 8]['gameid'].unique()

# 해당 gameId를 가진 행의 인덱스 추출 후 삭제
drop_idx_3 = df_all_match_2[df_all_match_2['gameid'].isin(drop_game_ids_3)].index
df_all_match_2 = df_all_match_2.drop(index=drop_idx_3)

print(f"player_cnt < 8인 데이터가 포함된 경기 데이터 삭제한 후: {df_all_match_2.shape}")

player_cnt < 8인 데이터가 포함된 경기 데이터 삭제한 후: (399872, 10)


### 3-6. 시즌2 데이터 삭제

In [66]:
# 시즌 2 고유 시너지 목록
season2_keys = {'Alchemist', 'Avatar', 'Berserker',
                'Crystal', 'Desert', 'Druid', 'Electric',
                'Light', 'Mage', 'Mountain', 'Mystic', 'Ocean',
                'Poison', 'Predator', 'Set2_Assassin', 'Set2_Blademaster',
                'Set2_Glacial', 'Set2_Ranger', 'Shadow', 'Soulbound',
                'Summoner', 'Warden', 'Wind', 'Woodland'}  # 시즌 2 시너지 입력


# 시즌 2 시너지가 하나라도 포함되면 시즌 2로 분류
df_all_match_2['season'] = df_all_match_2['combination'].apply(
    lambda x: 'season 2' if any(k in season2_keys for k in ast.literal_eval(x).keys())
    else 'season 3'
)

# season 2인 행의 gameId 추출
drop_game_ids_4 = df_all_match_2[df_all_match_2['season'] == 'season 2']['gameid'].unique()


# 해당 gameId를 가진 행의 인덱스 추출 후 삭제
drop_idx_4 = df_all_match_2[df_all_match_2['gameid'].isin(drop_game_ids_4)].index
df_all_match_2 = df_all_match_2.drop(index=drop_idx_4)

print(f"season 2인 데이터가 포함된 경기 데이터 삭제한 후: {df_all_match_2.shape}")

season 2인 데이터가 포함된 경기 데이터 삭제한 후: (396640, 11)


### 3-7. 시즌 3 시너지 더미 데이터 ”TemplateTrait” 삭제
- 시너지 데이터 중 해당 값인 `{'TemplateTrait': 1}`만 삭제

In [67]:
# TemplateTrait 키가 있는 행만 필터링 후 값 확인
df_all_match_2[df_all_match_2['combination'].apply(
    lambda x: 'TemplateTrait' in ast.literal_eval(x).keys() # TemplateTrait 키가 있는 행만 필터링
)]['combination'].apply(                                    # 필터링된 행 중에서 combination 컬럼만 가져옴
    lambda x: ast.literal_eval(x).get('TemplateTrait')      # TemplateTrait 키에 할당된 값을 가져옴
).value_counts()


combination
1    65987
Name: count, dtype: int64

In [68]:
# 분석 목적에 영향을 주지 않는 dummy 데이터로 판단하여 시너지 중 'TemplateTrait'만 삭제
df_all_match_2['combination'] = df_all_match_2['combination'].apply(
    lambda x: json.dumps(
        # key: value = 키-값의 쌍을 의미함
        {key: value for key, value in ast.literal_eval(x).items() if key != 'TemplateTrait'},
        ensure_ascii=False
    )
)

In [69]:
# TemplateTrait 키가 있는 행 수 확인 → (0,11)이면 완전히 삭제된 것
df_all_match_2[df_all_match_2['combination'].apply(
    lambda x: 'TemplateTrait' in ast.literal_eval(x).keys()
)].shape

(0, 11)

### 3-8. 유저 ID 컬럼 제작

In [70]:
df_all_match_2['user_id'] = 'KR-USER-' + (df_all_match_2.index + 1).astype(str)

In [71]:
df_all_match_2['user_id']

0              KR-USER-1
1              KR-USER-2
2              KR-USER-3
3              KR-USER-4
4              KR-USER-5
               ...      
399993    KR-USER-399994
399994    KR-USER-399995
399995    KR-USER-399996
399996    KR-USER-399997
399997    KR-USER-399998
Name: user_id, Length: 396640, dtype: str

### 3-9. 티어가 섞인 경기 정보 추출

In [72]:
# 티어가 섞인 gameId 추출
mixed_gameids = df_all_match_2.groupby('gameid')['tier'].nunique() # gameid별로 tier의 고유값 개수를 계산
mixed_gameids = mixed_gameids[mixed_gameids > 1].index # 고유값이 2 이상인 gameId만 추출

# 해당 gameId의 티어 구성 확인
# mixed_gameids에 포함된 gameId를 가진 행만 추출 -> gameid별로 어떤 티어가 몇 명씩 있는지 카운트
df_all_match_2[df_all_match_2['gameid'].isin(mixed_gameids)].groupby('gameid')['tier'].value_counts()

gameid         tier        
KR_4263820118  platinum        8
               master          8
KR_4313697214  master          8
               platinum        8
KR_4320079059  platinum        8
               diamond         8
KR_4344513862  master          8
               diamond         8
KR_4347884427  diamond         8
               master          8
KR_4357966612  grand_master    8
               platinum        8
KR_4358922415  master          8
               diamond         8
KR_4361594426  master          8
               diamond         8
KR_4361773461  diamond         8
               grand_master    8
KR_4362846604  master          8
               diamond         8
KR_4364453473  grand_master    8
               diamond         8
KR_4365284161  master          8
               diamond         8
KR_4378896137  diamond         8
               platinum        8
KR_4381231118  diamond         8
               platinum        8
KR_4387025966  diamond         8
               

In [73]:
# 티어 순서 정의 (숫자가 높을수록 높은 티어)
tier_order = {
    'platinum': 1,
    'diamond': 2,
    'master': 3,
    'grand_master': 4,
    'challenger': 5
}

---
## 3-10. 빈 딕셔너리 (결측 combination/champion) 감지 및 분석

### 문제 
- `gameId` 단위로 정상 데이터를 처리하는 과정에서, 정원 8명 중 일부 플레이어의 `combination`/`champion`이 빈 딕셔너리 `{}`로 기록되어 있음
- 한 게임의 8명 중 한 명이라도 문제가 있으면 전체 게임의 신뢰도 감소

### 처리 규칙
1. **둘 다 결측 (Combo='{}' AND Champ='{}')**: 해당 행이 포함된 게임 전체 삭제
2. **하나만 결측**:
   - `flag_1` (boolean): Combo O + Champ X → 사후검정용
   - `flag_2` (boolean): Combo X + Champ O → 복구용


In [74]:
# 딕셔너리 파싱 함수
def parse_dict(val):
    """문자열을 딕셔너리로 파싱"""
    try:
        if isinstance(val, str):
            return ast.literal_eval(val)
        return val
    except:
        return {}

In [75]:
def is_empty_dict(x):
    """딕셔너리가 비어있는지 확인"""
    try:
        parsed = parse_dict(x)
        return isinstance(parsed, dict) and len(parsed) == 0
    except:
        return False

print("함수 정의 완료")

함수 정의 완료


In [76]:
print("처리 전 상태 확인")
print(f"\n 데이터 기본 정보")
print(f"전체 행 수: {len(df_all_match_2):,}")
print(f"전체 게임 수: {df_all_match_2['gameid'].nunique():,}")

# 빈 딕셔너리 감지
empty_combo_count = 0
empty_champ_count = 0
both_empty_count = 0

for idx, row in df_all_match_2.iterrows():
    is_combo_empty = is_empty_dict(row['combination'])
    is_champ_empty = is_empty_dict(row['champion'])
    
    if is_combo_empty:
        empty_combo_count += 1
    if is_champ_empty:
        empty_champ_count += 1
    if is_combo_empty and is_champ_empty:
        both_empty_count += 1

print(f"\n결측 패턴")
print(f"Empty combination: {empty_combo_count:,} ({empty_combo_count/len(df_all_match_2)*100:.2f}%)")
print(f"Empty champion: {empty_champ_count:,} ({empty_champ_count/len(df_all_match_2)*100:.2f}%)")
print(f"둘 다 empty: {both_empty_count:,} ({both_empty_count/len(df_all_match_2)*100:.2f}%)")
print(f"둘 중 하나만 empty: {empty_combo_count + empty_champ_count - 2*both_empty_count:,}")

처리 전 상태 확인

 데이터 기본 정보
전체 행 수: 396,640
전체 게임 수: 49,561

결측 패턴
Empty combination: 380 (0.10%)
Empty champion: 298 (0.08%)
둘 다 empty: 263 (0.07%)
둘 중 하나만 empty: 152


In [77]:
print("둘 다 결측인 행이 포함된 게임 찾기")

games_to_delete = set()

for idx, row in df_all_match_2.iterrows():
    is_combo_empty = is_empty_dict(row['combination'])
    is_champ_empty = is_empty_dict(row['champion'])
    
    if is_combo_empty and is_champ_empty:
        games_to_delete.add(row['gameid'])

print(f"\n삭제 대상 게임")
print(f"둘 다 empty 행이 포함된 게임: {len(games_to_delete):,}개")


둘 다 결측인 행이 포함된 게임 찾기

삭제 대상 게임
둘 다 empty 행이 포함된 게임: 217개


In [78]:
print("플래그 컬럼 생성")

print("\n플래그 정의")
print("flag_1 (boolean): combination O + champion X (사후검정용)")
print("flag_2 (boolean): combination X + champion O (복구용)")

df_all_match_2['flag_1'] = False  # combination O + champion X
df_all_match_2['flag_2'] = False  # combination X + champion O

flag_1_count = 0
flag_2_count = 0

for idx, row in df_all_match_2.iterrows():
    is_combo_empty = is_empty_dict(row['combination'])
    is_champ_empty = is_empty_dict(row['champion'])
    
    # flag_1: combination O + champion X 
    if not is_combo_empty and is_champ_empty:
        df_all_match_2.at[idx, 'flag_1'] = True
        flag_1_count += 1
    
    # flag_2: combination X + champion O
    if is_combo_empty and not is_champ_empty:
        df_all_match_2.at[idx, 'flag_2'] = True
        flag_2_count += 1

print("\n플래그 컬럼 생성 결과")
print(f"flag_1 (Combo O + Champ X): {flag_1_count:,}행 ({flag_1_count/len(df_all_match_2)*100:.2f}%)")
print(f"flag_2 (Combo X + Champ O): {flag_2_count:,}행 ({flag_2_count/len(df_all_match_2)*100:.2f}%)")
print(f"둘 다 False: {len(df_all_match_2) - flag_1_count - flag_2_count:,}행")

플래그 컬럼 생성

플래그 정의
flag_1 (boolean): combination O + champion X (사후검정용)
flag_2 (boolean): combination X + champion O (복구용)

플래그 컬럼 생성 결과
flag_1 (Combo O + Champ X): 35행 (0.01%)
flag_2 (Combo X + Champ O): 117행 (0.03%)
둘 다 False: 396,488행


In [79]:
print("둘 다 결측인 행이 포함된 게임 삭제")

rows_before = len(df_all_match_2)
games_before = df_all_match_2['gameid'].nunique()

# 게임 삭제
df_all_match_3 = df_all_match_2[~df_all_match_2['gameid'].isin(games_to_delete)].reset_index(drop=True)

#확인용
rows_after = len(df_all_match_3)
games_after = df_all_match_3['gameid'].nunique()

print("\n[삭제 결과")
print(f"삭제 전 행 수: {rows_before:,}")
print(f"삭제 전 게임 수: {games_before:,}")
print(f"\n삭제된 행 수: {rows_before - rows_after:,} ({(rows_before - rows_after)/rows_before*100:.2f}%)")
print(f"삭제된 게임 수: {len(games_to_delete):,} ({len(games_to_delete)/games_before*100:.2f}%)")
print(f"\n삭제 후 행 수: {rows_after:,}")
print(f"삭제 후 게임 수: {games_after:,}")
print(f"데이터 보존율: {rows_after/rows_before*100:.2f}%")

둘 다 결측인 행이 포함된 게임 삭제

[삭제 결과
삭제 전 행 수: 396,640
삭제 전 게임 수: 49,561

삭제된 행 수: 1,736 (0.44%)
삭제된 게임 수: 217 (0.44%)

삭제 후 행 수: 394,904
삭제 후 게임 수: 49,344
데이터 보존율: 99.56%


In [80]:
print(f"\n빈 딕셔너리 제거 검증")

# 정제된 데이터에서 둘 다 empty인 행 확인
both_empty_remaining = 0
for idx, row in df_all_match_3.iterrows():
    is_combo_empty = is_empty_dict(row['combination'])
    is_champ_empty = is_empty_dict(row['champion'])
    
    if is_combo_empty and is_champ_empty:
        both_empty_remaining += 1

print(f"정제 후 '둘 다 empty' 행: {both_empty_remaining:,}개")
if both_empty_remaining == 0:
    print("검증 성공: 모든 '둘 다 empty' 행 제거 완료!")
else:
    print("검증 실패: 여전히 '둘 다 empty' 행이 존재합니다!")

print("\n플래그 컬럼 검증")
flag_1_count_after = df_all_match_3['flag_1'].sum()
flag_2_count_after = df_all_match_3['flag_2'].sum()

print(f"flag_1 = True인 행: {flag_1_count_after:,}개")
print(f"flag_2 = True인 행: {flag_2_count_after:,}개")
print(f"둘 다 False (정상): {len(df_all_match_3) - flag_1_count_after - flag_2_count_after:,}개")


빈 딕셔너리 제거 검증
정제 후 '둘 다 empty' 행: 0개
검증 성공: 모든 '둘 다 empty' 행 제거 완료!

플래그 컬럼 검증
flag_1 = True인 행: 29개
flag_2 = True인 행: 115개
둘 다 False (정상): 394,760개


In [81]:
print("\nflag_1 (Combo O + Champ X) 샘플 확인")

flag_1_rows = df_all_match_3[df_all_match_3['flag_1'] == True]
if len(flag_1_rows) > 0:
    print(f"총 {len(flag_1_rows):,}개 행이 flag_1=True\n")
    for idx, (_, row) in enumerate(flag_1_rows.head(3).iterrows(), 1):
        combo = parse_dict(row['combination'])
        champ = parse_dict(row['champion'])
        print(f"{idx}. gameid: {row['gameid']}")
        print(f"   combination: {len(combo)} items (정상) | champion: {len(champ)} items (EMPTY)")
        print()
else:
    print("flag_1=True인 행이 없습니다.")

print("\nflag_2 (Combo X + Champ O) 샘플 확인")
print("-"*80)

flag_2_rows = df_all_match_3[df_all_match_3['flag_2'] == True]
if len(flag_2_rows) > 0:
    print(f"총 {len(flag_2_rows):,}개 행이 flag_2=True\n")
    for idx, (_, row) in enumerate(flag_2_rows.head(3).iterrows(), 1):
        combo = parse_dict(row['combination'])
        champ = parse_dict(row['champion'])
        print(f"{idx}. gameid: {row['gameid']}")
        print(f"   combination: {len(combo)} items (EMPTY) | champion: {len(champ)} items (정상)")
        print()
else:
    print("flag_2=True인 행이 없습니다.")


flag_1 (Combo O + Champ X) 샘플 확인
총 29개 행이 flag_1=True

1. gameid: KR_4231588513
   combination: 4 items (정상) | champion: 0 items (EMPTY)

2. gameid: KR_4231588513
   combination: 5 items (정상) | champion: 0 items (EMPTY)

3. gameid: KR_4231588513
   combination: 9 items (정상) | champion: 0 items (EMPTY)


flag_2 (Combo X + Champ O) 샘플 확인
--------------------------------------------------------------------------------
총 115개 행이 flag_2=True

1. gameid: KR_4271274609
   combination: 0 items (EMPTY) | champion: 5 items (정상)

2. gameid: KR_4359182660
   combination: 0 items (EMPTY) | champion: 2 items (정상)

3. gameid: KR_4320179268
   combination: 0 items (EMPTY) | champion: 4 items (정상)



In [82]:
df_all_match_3

,gameid,gameduration,level,lastround,ranked,ingameduration,combination,champion,tier,player_cnt,season,user_id,flag_1,flag_2
0,KR_4291707834,1963.905273,6,27,5,1390.165771,"{""Cybernetic"": 1, ""Demolitionist"": 1, ""Infiltrator"": 1, ""Rebel"": 1, ""Set3_Brawler"": 1, ""Set3_Celestial"": 1, ""Set3_Void"": 1, ""Sniper"": 1}","{'Ziggs': {'items': [7], 'star': 1}, 'Ashe': {'items': [9], 'star': 1}, 'ChoGath': {'items': [6], 'star': 1}, 'Ekko': {'items': [1], 'star': 1}}",platinum,8,season 3,KR-USER-1,False,False
1,KR_4291707834,1963.905273,8,37,3,1891.282715,"{""Blaster"": 1, ""Chrono"": 1, ""Cybernetic"": 4, ""Demolitionist"": 1, ""Rebel"": 1, ""Set3_Blademaster"": 2, ""Set3_Brawler"": 1, ""Set3_Sorcerer"": 1, ""Set3_Void"": 1, ""Valkyrie"": 1, ""Vanguard"": 2}","{'Ziggs': {'items': [24], 'star': 3}, 'Fiora': {'items': [37], 'star': 2}, 'Leona': {'items': [36, 24], 'star': 2}, 'Lucian': {'items': [], 'star': 2}, 'Vi': {'items': [5], 'star': 2}, 'Kayle': {'items': [], 'star': 2}, 'WuKong': {'items': [3, 67], 'star': 2}, 'VelKoz': {'items': [4], 'star': 2}}",platinum,8,season 3,KR-USER-2,False,False
2,KR_4291707834,1963.905273,6,25,7,1279.461060,"{""Blaster"": 1, ""Cybernetic"": 1, ""DarkStar"": 2, ""Infiltrator"": 1, ""Mercenary"": 1, ""Set3_Blademaster"": 1, ""Set3_Mystic"": 1, ""Valkyrie"": 1}","{'Fiora': {'items': [1], 'star': 1}, 'Shaco': {'items': [6], 'star': 1}, 'Karma': {'items': [4], 'star': 1}, 'MissFortune': {'items': [3], 'star': 1}}",platinum,8,season 3,KR-USER-3,False,False
3,KR_4291707834,1963.905273,7,38,2,1955.608521,"{""DarkStar"": 1, ""Protector"": 2, ""Set3_Blademaster"": 1, ""Set3_Celestial"": 5, ""Set3_Mystic"": 1, ""Sniper"": 1, ""StarGuardian"": 2, ""Vanguard"": 2}","{'Poppy': {'items': [], 'star': 2}, 'Xayah': {'items': [19, 23, 19], 'star': 3}, 'Rakan': {'items': [], 'star': 2}, 'XinZhao': {'items': [16], 'star': 2}, 'Mordekaiser': {'items': [35, 67, 33], 'star': 3}, 'Ashe': {'items': [], 'star': 2}, 'Soraka': {'items': [68, 47], 'star': 2}}",platinum,8,season 3,KR-USER-4,False,False
4,KR_4291707834,1963.905273,8,38,1,1955.608521,"{""Blaster"": 1, ""Chrono"": 5, ""DarkStar"": 3, ""Protector"": 1, ""Set3_Blademaster"": 1, ""Set3_Brawler"": 1, ""Set3_Sorcerer"": 2, ""Sniper"": 2}","{'TwistedFate': {'items': [36, 27], 'star': 3}, 'Caitlyn': {'items': [49, 29], 'star': 2}, 'JarvanIV': {'items': [56], 'star': 2}, 'Blitzcrank': {'items': [15], 'star': 2}, 'Shen': {'items': [77, 6], 'star': 2}, 'Ezreal': {'items': [16], 'star': 2}, 'Lux': {'items': [], 'star': 2}, 'Jhin': {'items': [], 'star': 2}}",platinum,8,season 3,KR-USER-5,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
394899,KR_4357265434,2094.518555,7,33,4,1838.332764,"{""DarkStar"": 2, ""Demolitionist"": 2, ""Infiltrator"": 4, ""MechPilot"": 3, ""Set3_Sorcerer"": 2, ""Set3_Void"": 1, ""Valkyrie"": 1}","{'KhaZix': {'items': [67, 26, 56], 'star': 2}, 'KaiSa': {'items': [38, 44, 33], 'star': 2}, 'Annie': {'items': [], 'star': 3}, 'Shaco': {'items': [45, 29, 16], 'star': 3}, 'Rumble': {'items': [55, 66, 77], 'star': 2}, 'Lux': {'items': [46], 'star': 2}, 'Fizz': {'items': [], 'star': 2}}",challenger,8,season 3,KR-USER-399994,False,False
394900,KR_4357265434,2094.518555,8,33,5,1837.548096,"{""Blaster"": 1, ""Chrono"": 2, ""Cybernetic"": 6, ""Infiltrator"": 2, ""ManaReaver"": 2, ""Set3_Blademaster"": 3, ""Set3_Brawler"": 1, ""Vanguard"": 1}","{'Fiora': {'items': [35], 'star': 2}, 'Leona': {'items': [22], 'star': 1}, 'Shen': {'items': [], 'star': 1}, 'Lucian': {'items': [57, 2, 23], 'star': 2}, 'Vi': {'items': [67, 36], 'star': 2}, 'Irelia': {'items': [29, 28, 15], 'star': 2}, 'Thresh': {'items': [57], 'star': 2}, 'Ekko': {'items': [35, 15, 2], 'star': 2}}",challenger,8,season 3,KR-USER-399995,False,False
394901,KR_4357265434,2094.518555,9,33,6,1833.472046,"{""Chrono"": 2, ""Cybernetic"": 1, ""Demolitionist"": 2, ""Infiltrator"": 2, ""MechPilot"": 1, ""Mercenary"": 1, ""Rebel"": 1, ""Set3_Brawler"": 4, ""Set3_Sorcerer"": 2, ""Set3_Voi

In [ ]:
print("Flag 1, 2 게임 ID 리스트 추출")

# 플래그 확인
flag_1_mask = df_all_match_3['flag_1'] == True
flag_2_mask = df_all_match_3['flag_2'] == True

# 게임 ID 추출
flag_1_game_ids = set(df_all_match_3[flag_1_mask]['gameid'].unique())
flag_2_game_ids = set(df_all_match_3[flag_2_mask]['gameid'].unique())

print(f"\nFlag 게임 ID 개수")
print(f"flag_1 게임 ID 개수: {len(flag_1_game_ids):,}개")
print(f"flag_2 게임 ID 개수: {len(flag_2_game_ids):,}개")
print(f"두 플래그가 겹치는 게임: {len(flag_1_game_ids & flag_2_game_ids):,}개")

# 리스트로 변환
flag_1_game_ids_list = sorted(list(flag_1_game_ids))
flag_2_game_ids_list = sorted(list(flag_2_game_ids))


Flag 1, 2 게임 ID 리스트 추출

Flag 게임 ID 개수
flag_1 게임 ID 개수: 5개
flag_2 게임 ID 개수: 115개
두 플래그가 겹치는 게임: 0개


In [91]:
flag_2_game_ids_list

['KR_4271274609',
 'KR_4275570139',
 'KR_4280509972',
 'KR_4280775299',
 'KR_4282545658',
 'KR_4289338197',
 'KR_4289735897',
 'KR_4290691308',
 'KR_4294692860',
 'KR_4296351103',
 'KR_4297130798',
 'KR_4297845328',
 'KR_4299048416',
 'KR_4299905690',
 'KR_4300306742',
 'KR_4302185296',
 'KR_4303978194',
 'KR_4304139091',
 'KR_4305607733',
 'KR_4310913023',
 'KR_4311960180',
 'KR_4312432447',
 'KR_4313489084',
 'KR_4316731107',
 'KR_4316871401',
 'KR_4317019936',
 'KR_4318795367',
 'KR_4319063178',
 'KR_4319453241',
 'KR_4320179268',
 'KR_4320583924',
 'KR_4320825079',
 'KR_4321051966',
 'KR_4321916992',
 'KR_4323040528',
 'KR_4324633175',
 'KR_4326077114',
 'KR_4328925229',
 'KR_4328956024',
 'KR_4329179072',
 'KR_4329469775',
 'KR_4329943985',
 'KR_4329954266',
 'KR_4331830791',
 'KR_4332575237',
 'KR_4333355139',
 'KR_4333774624',
 'KR_4334218028',
 'KR_4334932041',
 'KR_4335172878',
 'KR_4335428064',
 'KR_4336530978',
 'KR_4336652699',
 'KR_4337925947',
 'KR_4338428849',
 'KR_43387